In [8]:
import matplotlib.patches as patches
from matplotlib.path import Path
import muspy
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import torch


# ===== 2. 绘图设置 =====
CMAP = plt.cm.get_cmap('tab20', 11)  # 自动配色
COLORS = [CMAP(i) for i in range(11)]
INSTR_MAP = [
    'Piano', 'Chromatic Percussion', 'Organ', 'Guitar', 'Bass', 'Strings', 'Ensemble', 'Brass', 'Reed', 'Pipe', 'Synth Effect'
]


def draw_wave(ax, x, y, color, is_long_note=True):
    """波浪线 / 短音时画成断开的波浪"""
    t = np.linspace(0, 1, 80)
    X = y + (t - 0.5) * 0.8
    Y = x + 0.3 * np.sin(2 * np.pi * t)

    if is_long_note:
        ax.plot(X, Y, color=color, alpha=0.6, linewidth=10.0)
    else:
        # 分段画，比如隔 10 个点断开一次
        step = 15
        for i in range(0, len(t)-step, step*2):
            ax.plot(X[i:i+step], Y[i:i+step],
                    color=color, alpha=0.6, linewidth=10.0)


def draw_round_rect(ax, x, y, color, is_long_note=True):
    """圆角矩形；短音符时在中间挖空两段"""
    # 整体矩形参数
    width, height = 0.8, 0.3
    base_rect = patches.FancyBboxPatch(
        (y - width/2, x - height/2),
        width, height,
        boxstyle="round,pad=0.1,rounding_size=0.2",
        linewidth=0,
        facecolor=color, alpha=0.5
    )
    ax.add_patch(base_rect)

    if not is_long_note:
        # 在矩形上叠加两段白色遮挡条，制造断开的效果
        gap_w, gap_h = 0.15, height + 0.2
        offsets = [-0.15, 0.15]  # 两个空白的 x 偏移
        for off in offsets:
            gap = patches.Rectangle(
                (y + off - gap_w/2, x - gap_h/2),
                gap_w, gap_h,
                linewidth=0,
                facecolor="white", edgecolor="none",

            )
            ax.add_patch(gap)


def plot_tonal_hyperscore(hyperscore: np.ndarray, fpath,xlim=(0.5,2.5),ylim=(1.5,3.5)):
    W, H, C = hyperscore.shape[0], hyperscore.shape[1], hyperscore.shape[2]

    

    # 为不同通道生成 (dx, dy) 偏移，避免重叠
    angles = np.linspace(0, 2*np.pi, C, endpoint=False)
    radius = 0.2
    offsets = [(radius*np.cos(a), radius*np.sin(a)) for a in angles]

    legend_handles = []
    # ===== 3. 绘制数据 =====
    for c in range(C):
        drawn = True
        fig, ax = plt.subplots(figsize=(8, 8))

        ax.set_aspect('equal')
        ax.set_ylim(ylim)
        ax.set_xlim(xlim)
        for w in range(W):
            for h in range(8):
                is_melodic, is_harmonic, is_long_note = hyperscore[w, h, c]

                # dx, dy = offsets[c]
                # x = h + dx
                # y = w + dy
                x=h
                y=w

                if is_melodic:  # 波浪线
                    draw_wave(ax, x, y, COLORS[c], is_long_note)
                    drawn = True

                if is_harmonic:  # 圆角矩形

                    draw_round_rect(ax, x, y, COLORS[c], is_long_note)
                    drawn = True
        if drawn:
            legend_handles.append(
                patches.Patch(color=COLORS[c], label=INSTR_MAP[c], alpha=0.5)
            )
    

        plt.tight_layout()
        plt.axis("off")
        plt.savefig(fpath + INSTR_MAP[c] + ".svg", dpi=300,bbox_inches="tight", pad_inches=0)
        plt.close(fig)



def plot_pianoroll_from_multitrack_pianoroll(pianoroll: np.ndarray, fpath, plot_mask=False, xlim=(0.0, 32.0),ylim=(32.0, 64.0),linewidth=1.0):
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)
    for w in range(128):
        for h in range(128):
            all_mask = True
            for c in range(11):
                if all_mask:
                    if pianoroll[w,h,c] != 3:
                        all_mask=False
                color = COLORS[c]
                if pianoroll[w, h, c] == 1:
                    dw = 1
                    while True:
                        if w+dw >= 128:
                            break
                        if pianoroll[w+dw, h, c] != 2:
                            break
                        dw += 1
                    ax.plot([w, w+dw], [h, h], color=color,
                            linewidth=linewidth, alpha=0.5)
            if all_mask and plot_mask:
                ax.plot([w, w+1], [h, h], color='black',
                            linewidth=linewidth, alpha=0.5)

    plt.tight_layout()
    plt.axis("off")
    plt.savefig(fpath, dpi=300,bbox_inches="tight", pad_inches=0)
    plt.close(fig)


/tmp/ipykernel_271792/349572051.py:11: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  CMAP = plt.cm.get_cmap('tab20', 11)  # 自动配色


In [9]:
import numpy as np

def valid_vitex(vitex:np.ndarray):

    summation = np.sum(vitex.reshape(8,8,-1), axis=2)
    for w in range(8):
        for h in range(8):
            if summation[w,h] >= 5.5:
                return False
    instr_summation = np.sum(vitex.transpose(0,1,3,2).reshape(-1,11), axis=0)
    instr_pre = np.zeros_like(instr_summation)
    instr_pre[instr_summation >= 1.0] = 1.0
    if np.sum(instr_pre) >= 5.0:
        return False

    melodic_long = 0
    harmonic_long = 0
    melodic_short = 0
    harmonic_short = 0
    for c in range(11):
        for w in range(8):
            for h in range(8):
                melodic, harmonic, long = vitex[w,h,c]
                if melodic > 0.5 and long > 0.5:
                    melodic_long += 1
                if harmonic > 0.5 and long > 0.5:
                    harmonic_long += 1
                if melodic > 0.5 and not long > 0.5:
                    melodic_short += 1 
                if harmonic > 0.5 and not long > 0.5:
                    harmonic_short += 1 
                if melodic_long >= 2.0 and harmonic_long >= 2.0 and melodic_short >= 2.0 and harmonic_short>= 2.0:
                    return True
    return False

In [ ]:
from utils.inference_utils import get_dataset
from dataloader import collate_fn_no_shift
from torch.utils.data import DataLoader
from utils.output_utils import chord_to_midi, multitrack_pianoroll_to_midi, plot_pianoroll_from_drum_pianoroll,plot_drum_hyperscore
import matplotlib.pyplot as plt
from tqdm import tqdm
test_dl = DataLoader(get_dataset(), 1, False, collate_fn=collate_fn_no_shift)
import os
import numpy as np
indices = [4988]
for idx, batch_segment in tqdm(enumerate(test_dl)):
    if idx >= 5000: break
    if idx not in indices: continue
    save_folder = os.path.join("figures",f"{idx}")
    
    chord = batch_segment[0][0]
    np.save(os.path.join(save_folder, "gt_chd.npy"), chord.cpu().numpy())
    chord_to_midi(chord.permute(1,0,2), os.path.join(save_folder,"gt_chd.mid"))
    
    instrumentation = batch_segment[3][0].cpu().numpy().transpose(1,2,0).reshape(8,8,11,3)
    os.makedirs(save_folder, exist_ok=True)

    plot_tonal_hyperscore(instrumentation, os.path.join(save_folder, "ins"), xlim=(-0.5,7.5),ylim=(0.0,8.0))
    np.save(os.path.join(save_folder, "gt_ins.npy"), instrumentation)

    drum_hyperscore = batch_segment[1][0].cpu().numpy().transpose(1,2,0)
    plot_drum_hyperscore(drum_hyperscore, os.path.join(save_folder,"drum_ins.svg"))
    np.save(os.path.join(save_folder, "drum_ins.npy"), drum_hyperscore)

    pr_np = batch_segment[4][0].cpu().numpy().transpose(1,2,0)
    drum_np = batch_segment[2][0].cpu().numpy().reshape(128, 128)
    plot_pianoroll_from_multitrack_pianoroll(pr_np, os.path.join(save_folder, "pr.svg"), xlim=(16, 16+63.5), ylim=(25, 90), linewidth=7)
    plot_pianoroll_from_drum_pianoroll(drum_np, os.path.join(save_folder, "drum.svg"))
    np.save(os.path.join(save_folder, "gt_pr.npy"), pr_np)
    np.save(os.path.join(save_folder, "gt_drum.npy"), drum_np)
    multitrack_pianoroll_to_midi(pr_np,drum_np,os.path.join(save_folder, "gt_w_drum.mid"))
    multitrack_pianoroll_to_midi(pr_np,None,os.path.join(save_folder, "gt_wo_drum.mid"))
    plt.close('all')
    

        

Loading 717 songs from /workspace/midi_pkl_test_new


100%|██████████| 717/717 [00:00<00:00, 11089.00it/s]
831it [00:03, 230.46it/s]


In [ ]:
import matplotlib.patches as patches
from matplotlib.path import Path
import muspy
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import torch


# ===== 2. 绘图设置 =====
CMAP = plt.cm.get_cmap('tab20', 11)  # 自动配色
COLORS = [CMAP(i) for i in range(11)]
INSTR_MAP = [
    'Piano', 'Chromatic Percussion', 'Organ', 'Guitar', 'Bass', 'Strings', 'Ensemble', 'Brass', 'Reed', 'Pipe', 'Synth Effect'
]


def draw_wave(ax, x, y, color, is_long_note=True):
    """波浪线 / 短音时画成断开的波浪"""
    t = np.linspace(0, 1, 80)
    X = y + (t - 0.5) * 0.8
    Y = x + 0.3 * np.sin(2 * np.pi * t)

    if is_long_note:
        ax.plot(X, Y, color=color, alpha=0.6, linewidth=10.0)
    else:
        # 分段画，比如隔 10 个点断开一次
        step = 15
        for i in range(0, len(t)-step, step*2):
            ax.plot(X[i:i+step], Y[i:i+step],
                    color=color, alpha=0.6, linewidth=10.0)


def draw_round_rect(ax, x, y, color, is_long_note=True):
    """圆角矩形；短音符时在中间挖空两段"""
    # 整体矩形参数
    width, height = 0.8, 0.3
    base_rect = patches.FancyBboxPatch(
        (y - width/2, x - height/2),
        width, height,
        boxstyle="round,pad=0.1,rounding_size=0.2",
        linewidth=0,
        facecolor=color, alpha=0.5
    )
    ax.add_patch(base_rect)

    if not is_long_note:
        # 在矩形上叠加两段白色遮挡条，制造断开的效果
        gap_w, gap_h = 0.15, height + 0.2
        offsets = [-0.15, 0.15]  # 两个空白的 x 偏移
        for off in offsets:
            gap = patches.Rectangle(
                (y + off - gap_w/2, x - gap_h/2),
                gap_w, gap_h,
                linewidth=0,
                facecolor="white", edgecolor="none",

            )
            ax.add_patch(gap)


def plot_tonal_hyperscore(hyperscore: np.ndarray, fpath,xlim=(0.5,2.5),ylim=(1.5,3.5)):
    W, H, C = hyperscore.shape[0], hyperscore.shape[1], hyperscore.shape[2]

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.set_aspect('equal')
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)

    # 为不同通道生成 (dx, dy) 偏移，避免重叠
    angles = np.linspace(0, 2*np.pi, C, endpoint=False)
    radius = 0.2
    offsets = [(radius*np.cos(a), radius*np.sin(a)) for a in angles]

    legend_handles = []
    # ===== 3. 绘制数据 =====
    for c in range(C):
        drawn = False
        for w in range(W):
            for h in range(8):
                is_melodic, is_harmonic, is_long_note = hyperscore[w, h, c]

                # dx, dy = offsets[c]
                # x = h + dx
                # y = w + dy
                x=h
                y=w

                if is_melodic:  # 波浪线
                    draw_wave(ax, x, y, COLORS[c], is_long_note)
                    drawn = True

                if is_harmonic:  # 圆角矩形

                    draw_round_rect(ax, x, y, COLORS[c], is_long_note)
                    drawn = True
        if drawn:
            legend_handles.append(
                patches.Patch(color=COLORS[c], label=INSTR_MAP[c], alpha=0.5)
            )
    

    plt.tight_layout()
    plt.axis("off")
    plt.savefig(fpath, dpi=300,bbox_inches="tight", pad_inches=0)
    plt.close(fig)



def plot_pianoroll_from_multitrack_pianoroll(pianoroll: np.ndarray, fpath, plot_mask=False, xlim=(0.0, 32.0),ylim=(32.0, 64.0),linewidth=20.0):
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)
    for w in range(128):
        for h in range(128):
            all_mask = True
            for c in range(11):
                if all_mask:
                    if pianoroll[w,h,c] != 3:
                        all_mask=False
                color = COLORS[c]
                if pianoroll[w, h, c] == 1:
                    dw = 1
                    while True:
                        if w+dw >= 128:
                            break
                        if pianoroll[w+dw, h, c] != 2:
                            break
                        dw += 1
                    ax.plot([w, w+dw], [h, h], color=color,
                            linewidth=linewidth, alpha=0.5)
            if all_mask and plot_mask:
                ax.plot([w, w+1], [h, h], color='black',
                            linewidth=linewidth, alpha=0.5)

    plt.tight_layout()
    plt.axis("off")
    plt.savefig(fpath, dpi=300,bbox_inches="tight", pad_inches=0)
    plt.close(fig)


In [ ]:
test_hyperscore = np.zeros((8,8,11,3), dtype=bool)

# melodic and long
test_hyperscore[:,1,4,0] = True
test_hyperscore[:,1,4,1] = False
test_hyperscore[:,1,4,2] = True

# harmonic and long
test_hyperscore[:,2,6,0] = False
test_hyperscore[:,2,6,1] = True
test_hyperscore[:,2,6,2] = True

#harmonic and short
test_hyperscore[:,3,0,0] = False
test_hyperscore[:,3,0,1] = True
test_hyperscore[:,3,0,2] = False

# melodic and short
test_hyperscore[:,4,1,0] = True
test_hyperscore[:,4,1,1] = False
test_hyperscore[:,4,1,2] = False

In [ ]:
import torch
from config import UNetConfig, TrainingConfig
import numpy as np

import torch.nn.functional as F
import torch.nn as nn

from d3pm import D3PM

import os
import torch
from torch.utils.data import DataLoader
from accelerate import Accelerator
from torch import optim
from model.dual_unet import DualUNet
from tqdm import tqdm
from config import UNetConfig, TrainingConfig
from dataloader import get_train_dataloader
from utils.chord_utils import chords_to_sequence
from dataset import HyperscoreDataset


unet_config = UNetConfig(
    x1_shape=(11, 128, 128),
    x2_shape=(1, 128, 128),
    d_temb=256,
    hyperscore1_shape=(33, 8, 8),
    chord1_shape=(3, 32, 12),
    hyperscore2_shape=(2, 8, 3),
    base_channel=32,
    channel_mult=[1, 2, 4, 4],
    num_blocks=2,
    hyperscore1_level=[2, 3],
    hyperscore2_level=[2, 3],
    chord_level=[1, 2],
    fuse_rhythm_level=[],
    self_attn_level=[2, 3],
    num_heads=4,
    N=4,
    use_d3pm=True
)


training_config = TrainingConfig(
    batch_size=24,
    lr=5e-4,
    num_epochs=20,
    num_workers=10,
    pin_memory=True,
    accumulation_step=5,
    output_path="output",
    data_path="lmd_pkl",
    save_interval=20000,
    log_interval=100,
    null_condition_prob=0.5
)

# 2. 初始化 dummy optimizer 和 scheduler，用于恢复
dataset = HyperscoreDataset("/workspace/midi_pkl_test")
# 2. Load model
model = DualUNet(**vars(unet_config))

optimizer = optim.AdamW(model.parameters(), lr=training_config.lr)

d3pm = D3PM(model, 1000, num_classes=4, hybrid_loss_coeff=0.01)
accelerator = Accelerator()
# Prepare everything with accelerator
model, d3pm, optimizer, _ = accelerator.prepare(
    model, d3pm, optimizer, []
)
accelerator.load_state(
    "/workspace/hyperscore/output/20250909-070831/ckpt")  # 改成你实际路径
model = accelerator.unwrap_model(model)
model.eval()
d3pm: D3PM = accelerator.unwrap_model(d3pm)
d3pm.eval()
print("================Model Ready==================")
# 4. 准备条件
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   
example_chords2 = ['Am', 'Am', 'F', 'F',
                    'C', 'C', 'G', 'G',
                    'Am', 'Am', 'F', 'F',
                    'C', 'C', 'G', 'G',
                    'Am', 'Am', 'F', 'F',
                    'C', 'C', 'G', 'G',
                    'Am', 'Am', 'F', 'F',
                    'C', 'C', 'G', 'G',]
sequence2 = chords_to_sequence(example_chords2, beats_per_bar=4, bars=8)

cond_chords = [sequence2] 
print("=============condition chord ready====================")

cond_hyperscores = []
idx=600
sample = dataset[idx]
drum_hyperscore = sample.drum_hyperscore
tonal_hyperscore = test_hyperscore
plot_tonal_hyperscore(tonal_hyperscore, f"figures/tonal_hyperscore_{idx}.svg", xlim=(-0.5, 7.5), ylim=(0.0, 8.0))
cond_hyperscores.append((torch.from_numpy(tonal_hyperscore.transpose(2,3,0,1).reshape(33,8,8)).float(),
                            torch.from_numpy(drum_hyperscore.transpose(2,0,1)).float()))



In [ ]:

    

print("=============condition hyperscore ready==================")
print("Generating...")
num_gen = 1

stride = 100
for i1, cond_chord in enumerate(cond_chords):
    cond_chord = cond_chord[None, :, :, :].to(device)
    for i2, cond_hyperscore in enumerate(cond_hyperscores):
        tonal_hyperscore, drum_hyperscore = cond_hyperscore
        tonal_hyperscore = tonal_hyperscore[None, :, :, :].to(device)
        drum_hyperscore = drum_hyperscore[None, :, :, :].to(device)
        for n_gen in range(num_gen):
            sub_path = f"demo/d3pm/chord_{i1}_hyperscore_{i2}_{n_gen}"
            os.makedirs(sub_path, exist_ok=True)

            init_noise_tonal_pianoroll = 3*torch.ones((1, 11, 128, 128)
                                        ).to(device).to(torch.long)
            init_noise_drum_pianoroll = 3*torch.ones((1, 1, 128, 128)
                                        ).to(device).to(torch.long)
            x1, x2, x1s, x2s = d3pm.sample(
                init_noise_tonal_pianoroll, init_noise_drum_pianoroll, tonal_hyperscore, drum_hyperscore, cond_chord,
                    chord_scale=0.0, hyperscore_scale=1.1, x_inp=None, mask=None, return_all=True
            )

            # x1_np = x1.cpu().numpy()[0]
            # x1_np = x1_np.transpose(1,2,0)
            # x2_np = x2.cpu().numpy()[0]
            # x2_np = x2_np.reshape(128, 128)
            # multitrack_pianoroll_to_midi(x1_np, x2_np, f"{sub_path}/final.mid")
            # plot_pianoroll_from_multitrack_pianoroll(x1_np, f"{sub_path}/final_tonal_pianoroll.png")
            # plot_pianoroll_from_drum_pianoroll(x2_np, f"{sub_path}/final_drum_pianoroll.png")



In [ ]:
print(len(x1s), len(x2s))

In [ ]:
for idx, x1 in enumerate(x1s):
    if idx % 10 == 9:
        x1_np = x1.cpu().numpy()[0]
        x1_np = x1_np.transpose(1,2,0)
        plot_pianoroll_from_multitrack_pianoroll(x1_np, f"figures/{idx}.svg", plot_mask=False, ylim=(20.0, 128.0), xlim=(0.0, 128.0),linewidth=2.0)

In [ ]:
plot_pianoroll_from_multitrack_pianoroll(np.zeros_like(x1_np),f"figures/zero.svg")